In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import norm, skewnorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression
from scipy.stats import rankdata
import warnings, os, time

warnings.filterwarnings("ignore")

MODEL_NAME = "HPB"
DATASET_NAME = "Concrete"
RESULTS_DIR = "results"
PLOTS_DIR = "plots"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SPLIT_SEED = 58
SEEDS = [42, 58, 123, 256, 789]
N_RUNS = len(SEEDS)
DEVICE = torch.device("cpu")
torch.set_num_threads(1)

QUANTILES = [
    0.01, 0.025, 0.03, 0.05, 0.09, 0.10, 0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90, 0.91, 0.93, 0.95, 0.97, 0.975, 0.99,
]
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
CONTAMINATION_LEVELS = [0.02, 0.05, 0.10]
EP = 100; H = 100; LR = 0.01; BS = 32

plt.rcParams.update({
    "figure.figsize": (12, 5), "font.size": 11, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
})
C_PRED = "#2563EB"; C_TRUE = "#DC2626"; C_CLEAN = "#3B82F6"
C_CONTAM = "#F59E0B"; C_MODEL = "#7C3AED"; C_BAR = "#0EA5E9"

print(f"Config: {MODEL_NAME} | {DATASET_NAME} | {N_RUNS} seeds | BS={BS} | Device: {DEVICE}")


Config: HPB | Concrete | 5 seeds | BS=32 | Device: cpu


In [2]:
# ── Data Loading: Concrete ──
try:
    from sklearn.datasets import fetch_openml
    data = fetch_openml(name="Concrete_Compressive_Strength", version=1, as_frame=True, parser="auto")
    df = data.frame
    X = df.iloc[:, :-1].values.astype(float)
    y = df.iloc[:, -1].values.astype(float)
except Exception:
    try:
        data = fetch_openml(data_id=4353, as_frame=True, parser="auto")
        df = data.frame
        X = df.iloc[:, :-1].values.astype(float)
        y = df.iloc[:, -1].values.astype(float)
    except Exception:
        df = pd.read_csv("concrete.csv")
        X = df.iloc[:, :-1].values.astype(float)
        y = df.iloc[:, -1].values.astype(float)

scaler = StandardScaler()
X = scaler.fit_transform(X)
DIM = X.shape[1]
print(f"{DATASET_NAME}: {X.shape[0]} samples, {DIM} features")

Xtv, X_test, ytv, y_test = train_test_split(X, y, test_size=0.20, random_state=SPLIT_SEED)
X_train, X_val, y_train_clean, y_val = train_test_split(Xtv, ytv, test_size=0.20, random_state=SPLIT_SEED)
print(f"Split: Train={X_train.shape[0]} Val={X_val.shape[0]} Test={X_test.shape[0]}")

Concrete: 1030 samples, 8 features
Split: Train=659 Val=165 Test=206


In [3]:
def inject_outliers(y, frac, method, seed=None):
    rng = np.random.RandomState(seed)
    y = y.copy()
    n = len(y)
    n_out = max(1, int(frac * n))
    idx = rng.choice(n, n_out, replace=False)
    mu, sig = y.mean(), y.std()
    if method == "gaussian":
        y[idx] = rng.normal(mu, np.sqrt(5) * sig, n_out)
    elif method == "multiply":
        y[idx] = y[idx] * rng.choice([-1, 1], n_out) * 10
    elif method == "uniform":
        y[idx] = rng.uniform(y.min() - 3 * sig, y.max() + 3 * sig, n_out)
    elif method == "skewed":
        y[idx] = skewnorm.rvs(a=10, loc=mu, scale=3 * sig, size=n_out, random_state=rng)
    return y

cont_sets = {}
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        cont_sets[(ot, cl)] = inject_outliers(y_train_clean, cl, ot, seed=SPLIT_SEED)
print(f"{len(cont_sets)} contaminated training sets created")


12 contaminated training sets created


In [4]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, H), nn.ReLU(), nn.Linear(H, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()

def train_model(X, y, loss_fn, epochs=EP):
    model = MLP(DIM).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=LR)
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
    model.eval()
    return model

def train_mse(X, y, epochs=EP):
    return train_model(X, y, lambda p, t: nn.functional.mse_loss(p, t), epochs)

def pinball_loss(pred, target, tau):
    e = target - pred
    return torch.mean(torch.max(tau * e, (tau - 1) * e))

def hpb_loss(pred, target, tau, delta):
    r = target - pred
    ar = torch.abs(r)
    quadratic = 0.5 * r ** 2
    linear = ar * delta - 0.5 * delta ** 2
    base = torch.where(ar <= delta, quadratic, linear)
    weight = torch.where(r >= 0, tau, 1.0 - tau)
    return torch.mean(base * weight)

DELTA_GRID = [0.5, 1.0, 1.345, 1.5, 2.0]
print("HPB defined")


HPB defined


In [5]:
def compute_shift_rmse(preds_clean, preds_contam, quantiles):
    results = {}
    for tau in quantiles:
        diff = preds_clean[tau] - preds_contam[tau]
        results[tau] = np.sqrt(np.mean(diff ** 2))
    return results

def compute_ece(y_test, preds, quantiles):
    results = {}
    for tau in quantiles:
        coverage = np.mean(y_test <= preds[tau])
        results[tau] = abs(coverage - tau)
    return results
print("Evaluation functions defined")


Evaluation functions defined


In [6]:
all_preds = {}; all_shift_rmse = {}; all_ece_clean = {}; all_ece_contam = {}
all_best_deltas = {}
DELTA_GRID = [0.5, 1.0, 1.345, 1.5, 2.0]
conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]
print(f"δ grid: {DELTA_GRID}"); t0 = time.time()

X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
y_val_t = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)

for si, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSeed {seed} ({si+1}/{N_RUNS})\n{'='*60}")
    all_preds[seed] = {}; all_shift_rmse[seed] = {}; all_ece_contam[seed] = {}
    all_best_deltas[seed] = {}
    for ci, cond in enumerate(conditions):
        cond_label = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        y_tr = y_train_clean if cond == "clean" else cont_sets[cond]
        all_best_deltas[seed][cond] = {}
        preds = {}
        for tau in QUANTILES:
            best_delta, best_vl, best_m = None, float("inf"), None
            for delta in DELTA_GRID:
                set_seed(seed)
                m = train_model(X_train, y_tr, lambda p, y, t=tau, d=delta: hpb_loss(p, y, t, d))
                with torch.no_grad():
                    vl = pinball_loss(m(X_val_t), y_val_t, tau).item()
                if vl < best_vl:
                    best_vl = vl; best_delta = delta; best_m = m
            all_best_deltas[seed][cond][tau] = best_delta
            preds[tau] = predict(best_m, X_test)
        all_preds[seed][cond] = preds
        if cond == "clean":
            all_ece_clean[seed] = compute_ece(y_test, preds, QUANTILES)
        else:
            all_shift_rmse[seed][cond] = compute_shift_rmse(all_preds[seed]["clean"], preds, QUANTILES)
            all_ece_contam[seed][cond] = compute_ece(y_test, preds, QUANTILES)
        print(f"  [{(ci+1)/len(conditions)*100:5.1f}%] {cond_label:25s} | {(time.time()-t0)/60:.1f}min")
print(f"\nDone: {(time.time()-t0)/60:.1f} min")


δ grid: [0.5, 1.0, 1.345, 1.5, 2.0]

Seed 42 (1/5)


  [  7.7%] clean                     | 2.7min


  [ 15.4%] gaussian_2pct             | 5.1min


  [ 23.1%] gaussian_5pct             | 7.5min


  [ 30.8%] gaussian_10pct            | 9.9min


...

  [ 69.2%] uniform_5pct              | 148.1min


  [ 76.9%] uniform_10pct             | 150.5min


  [ 84.6%] skewed_2pct               | 153.0min


  [ 92.3%] skewed_5pct               | 155.4min


  [100.0%] skewed_10pct              | 157.8min

Done: 157.8 min


In [7]:
from openpyxl import Workbook
wb = Workbook()
ws_summary = wb.active
ws_summary.title = "Summary"
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ShiftRMSE_seed_{seed}")
    header = ["Condition"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_shift_rmse[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_clean_seed_{seed}")
    ws.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
    vals = all_ece_clean[seed]
    row = ["ECE_clean"] + [round(vals[t], 6) for t in QUANTILES]
    row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
    ws.append(row)

for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_contam_seed_{seed}")
    ws.append(["Condition"] + [str(t) for t in QUANTILES] + ["Avg"])
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_ece_contam[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

ws_summary.append(["=== Shift-RMSE (mean across seeds) ==="])
ws_summary.append(["Condition"] + [str(t) for t in QUANTILES] + ["Avg"])
for cond in contam_conditions:
    label = f"{cond[0]}_{int(cond[1]*100)}pct"
    means = [np.mean([all_shift_rmse[s][cond][t] for s in SEEDS]) for t in QUANTILES]
    ws_summary.append([label] + [round(m, 6) for m in means] + [round(np.mean(means), 6)])

ws_summary.append([])
ws_summary.append(["=== ECE Clean (mean across seeds) ==="])
ws_summary.append(["Metric"] + [str(t) for t in QUANTILES] + ["Avg"])
means_ece = [np.mean([all_ece_clean[s][t] for s in SEEDS]) for t in QUANTILES]
ws_summary.append(["ECE_clean"] + [round(m, 6) for m in means_ece] + [round(np.mean(means_ece), 6)])

ws_alpha = wb.create_sheet(title="Alpha")
ws_alpha.append(["Seed", "Condition", "Tau", "Best_Delta"])
for seed in SEEDS:
    for cond in ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]:
        label = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        for tau in QUANTILES:
            ws_alpha.append([f"seed_{seed}", label, tau, all_best_deltas[seed][cond].get(tau, "N/A")])

xlsx_path = f"{RESULTS_DIR}/{DATASET_NAME}_{MODEL_NAME}_results.xlsx"
wb.save(xlsx_path)
print(f"Saved: {xlsx_path}")


Saved: results/Concrete_HPB_results.xlsx


In [8]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in contam_conditions:
    cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
    fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
    per_seed = np.array([[all_shift_rmse[s][cond][t] for t in QUANTILES] for s in SEEDS])
    means, stds = per_seed.mean(0), per_seed.std(0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color=C_BAR, lw=2, markersize=5, capsize=3)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
    ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_avg.png"); plt.close()

    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = [all_shift_rmse[seed][cond][t] for t in QUANTILES]
        ax.plot(x_ticks, vals, "o-", color=C_BAR, lw=2, markersize=5)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
        ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_seed{seed}.png"); plt.close()
print("Shift-RMSE plots saved")


Shift-RMSE plots saved


In [9]:
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]
all_ece_conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in all_ece_conditions:
    if cond == "clean":
        cond_label, fname_tag = "Clean", "clean"
        per_seed = np.array([[all_ece_clean[s][t] for t in QUANTILES] for s in SEEDS])
    else:
        cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
        fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
        per_seed = np.array([[all_ece_contam[s][cond][t] for t in QUANTILES] for s in SEEDS])
    means, stds = per_seed.mean(0), per_seed.std(0)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color="#10B981", lw=2, markersize=5, capsize=3)
    ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
    ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label})")
    plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_avg.png"); plt.close()

    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = all_ece_clean[seed] if cond == "clean" else all_ece_contam[seed][cond]
        ax.plot(x_ticks, [vals[t] for t in QUANTILES], "o-", color="#10B981", lw=2, markersize=5)
        ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
        ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout(); plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_seed{seed}.png"); plt.close()
print("ECE plots saved")


ECE plots saved


In [10]:
print(f"{'='*60}")
print(f"{MODEL_NAME} on {DATASET_NAME} — Complete")
print(f"{'='*60}")
print(f"Results: {RESULTS_DIR}/"); print(f"Plots: {PLOTS_DIR}/")

print(f"\n--- Avg ECE (clean) ---")
avg_ece = np.mean([np.mean([all_ece_clean[s][t] for t in QUANTILES]) for s in SEEDS])
print(f"  Overall: {avg_ece:.4f}")

print(f"\n--- Avg Shift-RMSE (multiply 10%) ---")
for tau in [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]:
    vals = [all_shift_rmse[s][("multiply", 0.10)][tau] for s in SEEDS]
    print(f"  τ={tau:.3f}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")


HPB on Concrete — Complete
Results: results/
Plots: plots/

--- Avg ECE (clean) ---
  Overall: 0.0409

--- Avg Shift-RMSE (multiply 10%) ---
  τ=0.025: 278.1763 ± 5.7325
  τ=0.050: 133.0613 ± 2.5909
  τ=0.100: 14.6866 ± 2.6253
  τ=0.500: 3.1138 ± 0.4527
  τ=0.900: 79.0570 ± 11.2189
  τ=0.950: 211.7809 ± 1.9638
  τ=0.975: 295.3694 ± 3.7516
